In [ ]:
!pip install cupy 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 26.2 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
import os
import sys

# Clone the repository if it doesn't exist, or pull the latest changes
if not os.path.exists('/content/AutoDiff-Numpy'):
    !git clone -b v2 https://github.com/Seydifa/AutoDiff-Numpy.git /content/AutoDiff-Numpy
else:
    %cd /content/AutoDiff-Numpy
    !git pull
    %cd /content

# Add the cloned directory to sys.path so we can import dnp
if '/content/AutoDiff-Numpy' not in sys.path:
    sys.path.insert(0, '/content/AutoDiff-Numpy')

import dnp
print("✅ Successfully cloned and imported dnp!")


Cloning into '/content/AutoDiff-Numpy'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 136 (delta 37), reused 129 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 1.79 MiB | 25.50 MiB/s, done.
Resolving deltas: 100% (37/37), done.
✅ Successfully cloned and imported dnp!


In [2]:
import numpy as np
import dnp

def matmul_vjp_robust(g, x, y):
    """A robust VJP for np.matmul that properly handles N-D arrays and broadcasting."""
    # Transpose only the last two dimensions!
    dx = np.matmul(g, y.swapaxes(-1, -2))
    dy = np.matmul(x.swapaxes(-1, -2), g)
    
    # If parameters like Weights (2D) were broadcasted to match Batch dimensions (3D+),
    # we MUST sum out the extra batch dimensions to route the gradients properly.
    while dx.ndim > x.ndim:
        dx = dx.sum(axis=0)
    for i in range(x.ndim):
        if x.shape[i] == 1 and dx.shape[i] > 1:
            dx = dx.sum(axis=i, keepdims=True)
            
    while dy.ndim > y.ndim:
        dy = dy.sum(axis=0)
    for i in range(y.ndim):
        if y.shape[i] == 1 and dy.shape[i] > 1:
            dy = dy.sum(axis=i, keepdims=True)
            
    return dx, dy

# Overwrite the broken rule!
dnp.core.vjp_rules.VJP_RULES[np.matmul] = matmul_vjp_robust
print("✅ Patched np.matmul VJP rule for 3D/4D Tensor support!")

✅ Patched np.matmul VJP rule for 3D/4D Tensor support!


In [3]:
import os
import requests
import numpy as np

# Download data if missing
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
if not os.path.exists('input.txt'):
    with open('input.txt', 'w') as f:
        f.write(requests.get(url).text)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
data = np.array(encode(text), dtype=np.int32)

n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

batch_size = 4
block_size = 8
d_model = 32

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = np.random.randint(len(data_source) - block_size, size=(batch_size,))
    x_np = np.stack([data_source[i:i+block_size] for i in ix])
    y_np = np.stack([data_source[i+1:i+block_size+1] for i in ix])
    return x_np, y_np


In [4]:
class LayerNorm(dnp.core.layers.Module):
    def __init__(self, ndim, bias=True, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = dnp.core.Tensor(np.ones(ndim), name="LN_gamma")
        self.bias = bias
        if bias:
            self.beta = dnp.core.Tensor(np.zeros(ndim), name="LN_beta")

    def forward(self, x):
        mean = dnp.core.ops.mean(x, axis=-1, keepdims=True)
        diff = dnp.core.ops.subtract(x, mean)
        sq_diff = dnp.core.ops.square(diff)
        var = dnp.core.ops.mean(sq_diff, axis=-1, keepdims=True)
        
        # ✅ FIX HERE: Wrap eps in a Tensor to prevent missing 'y' gradient drop!
        eps_t = dnp.core.Tensor(np.array([self.eps], dtype=np.float32))
        std = dnp.core.ops.sqrt(dnp.core.ops.add(var, eps_t))
        
        x_norm = dnp.core.ops.divide(diff, std)
        out = dnp.core.ops.multiply(x_norm, self.gamma)
        if self.bias:
            out = dnp.core.ops.add(out, self.beta)
        return out

class FeedForward(dnp.core.layers.Module):
    def __init__(self, d_model):
        super().__init__()
        self.l1 = dnp.core.layers.Linear(d_model, 4 * d_model)
        self.relu = dnp.core.layers.ReLU()
        self.l2 = dnp.core.layers.Linear(4 * d_model, d_model)
        
    def forward(self, x):
        return self.l2(self.relu(self.l1(x)))

class TransformerBlock(dnp.core.layers.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.ln1 = LayerNorm(d_model)
        self.attn = dnp.core.layers.MultiHeadAttention(d_model, num_heads)
        self.ln2 = LayerNorm(d_model)
        self.ffwd = FeedForward(d_model)

    def forward(self, x, mask=None):
        x_ln1 = self.ln1(x)
        attn_out = self.attn(x_ln1, x_ln1, x_ln1, mask=mask)
        x = dnp.core.ops.add(x, attn_out)

        x_ln2 = self.ln2(x)
        ffwd_out = self.ffwd(x_ln2)
        x = dnp.core.ops.add(x, ffwd_out)
        return x


In [5]:
class TransformerLanguageModel(dnp.core.layers.Module):
    def __init__(self, vocab_size, d_model, block_size, num_heads, num_layers):
        super().__init__()
        
        # Using YOUR newly added Embedding!
        self.tok_emb = dnp.core.layers.Embedding(vocab_size, d_model, name="TokEmb")
        self.pos_emb = dnp.core.layers.Embedding(block_size, d_model, name="PosEmb")
        
        self.blocks = []
        for i in range(num_layers):
            block = TransformerBlock(d_model, num_heads)
            self._modules[f"block_{i}"] = block
            self.blocks.append(block)
            
        self.ln_f = LayerNorm(d_model)
        self.lm_head = dnp.core.layers.Linear(d_model, vocab_size, bias=False, name="LMHead")

    def forward(self, idx):
        B, T = np.asarray(idx).shape
        
        tok_emb = self.tok_emb(idx) 
        positions = np.arange(T)[None, :]
        pos_emb = self.pos_emb(positions) 
        
        x = dnp.core.ops.add(tok_emb, pos_emb)
        causal_mask = np.triu(np.ones((T, T)) * -1e9, k=1)
        t_mask = dnp.core.Tensor(causal_mask)
        
        for block in self.blocks:
            x = block(x, mask=t_mask)
            
        x = self.ln_f(x)
        return self.lm_head(x)

# Initialize model
model = TransformerLanguageModel(vocab_size, d_model, block_size, num_heads=4, num_layers=2)
print("Model initialized!")


Model initialized!


In [6]:
def cross_entropy_loss(logits, targets):
    B, T, V = logits.shape
    logits_flat = dnp.core.ops.reshape(logits, newshape=(B * T, V))
    targets_flat = targets.flatten()
    
    probs = dnp.core.ops.softmax(logits_flat)
    eps_tensor = dnp.core.Tensor(np.array([1e-8], dtype=np.float32))
    log_probs = dnp.core.ops.log(dnp.core.ops.add(probs, eps_tensor))
    
    one_hot = np.eye(V, dtype=np.float32)[targets_flat]
    t_one_hot = dnp.core.Tensor(one_hot)
    
    selected = dnp.core.ops.multiply(log_probs, t_one_hot)
    summed = dnp.core.ops.sum(selected, axis=-1)
    return dnp.core.ops.negative(dnp.core.ops.mean(summed))

# Initialize optimizer
optimizer = dnp.core.optimizers.AdamW(model.parameters(), lr=5e-3, weight_decay=1e-4)

print("--- Starting Training Loop Test ---")
for step in range(10):
    xb, yb = get_batch('train')
    logits = model(xb)
    loss = cross_entropy_loss(logits, yb)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Step {step} | Loss: {float(np.asarray(loss.data)):.4f}")


print("✅ Training step test successful! The loss is decreasing.")


--- Starting Training Loop Test ---


Step 0 | Loss: 4.8277
Step 1 | Loss: 4.5037
Step 2 | Loss: 4.3189
Step 3 | Loss: 4.2404
Step 4 | Loss: 3.8164
Step 5 | Loss: 4.0037
Step 6 | Loss: 3.7268
Step 7 | Loss: 3.7600
Step 8 | Loss: 3.9517
Step 9 | Loss: 3.9599
✅ Training step test successful! The loss is decreasing.


In [7]:
def generate(model, idx, max_new_tokens, block_size):
    # Ensure model is in eval mode (though we don't have dropout yet, good practice)
    model.eval()
    
    print(f"--- Generating {max_new_tokens} tokens ---")
    for _ in range(max_new_tokens):
        # Crop context if it exceeds block_size (Transformer context window limit)
        idx_cond = idx[:, -block_size:] 
        
        # Get predictions
        logits = model(idx_cond)
        
        # Focus only on the last time step prediction: (B, T, V) -> (B, V)
        logits_last = np.asarray(logits.data)[:, -1, :] 
        
        # Apply softmax to get probabilities
        v_max = np.max(logits_last, axis=-1, keepdims=True)
        exp_logits = np.exp(logits_last - v_max)
        probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        
        # Sample from the distribution
        p = probs[0]
        next_token = np.random.choice(len(p), p=p)
        
        # Append to the sequence
        idx = np.append(idx, [[next_token]], axis=1)
        
    return idx

# Generate 100 characters!
context = np.zeros((1, 1), dtype=np.int32) # Starting with a single newline char (0)
generated_indices = generate(model, context, max_new_tokens=100, block_size=block_size)

# Decode indices to text
decode = lambda l: ''.join([chars[i] for i in l])
generated_text = decode(generated_indices[0])

print("\\n--- GENERATED TEXT ---\\n")
print(generated_text)

--- Generating 100 tokens ---
\n--- GENERATED TEXT ---\n

e-Y t  oaaYdoCe-le?W.adnn!uNefu,wenahr. 3'  voOiy,xdTE rLZBflhJ.FoUmkO:MfW  a-rf!hlm uae -slms.uddoa


In [8]:
import time
from tqdm.auto import tqdm
from dnp.core.session import session

# --- 1. Scale Up Hyperparameters ---
batch_size = 32       
block_size = 16       
d_model = 64          
num_heads = 4
num_layers = 2
max_steps = 1000      

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = np.random.randint(len(data_source) - block_size, size=(batch_size,))
    x_np = np.stack([data_source[i:i+block_size] for i in ix])
    y_np = np.stack([data_source[i+1:i+block_size+1] for i in ix])
    return x_np, y_np

# --- 2. Re-initialize Larger Model & Optimizer ---
print("--- Initializing Larger Model ---")
big_model = TransformerLanguageModel(vocab_size, d_model, block_size, num_heads, num_layers)
optimizer = dnp.core.optimizers.AdamW(big_model.parameters(), lr=2e-3, weight_decay=1e-4)

# --- 3. Extended Training Loop with TQDM and Session Reset ---
print("--- Starting Extended Training ---")

pbar = tqdm(range(max_steps), desc="Training")

for step in pbar:
    xb, yb = get_batch('train')
    logits = big_model(xb)
    loss = cross_entropy_loss(logits, yb)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # ✅ CRITICAL: Clear the computation graph from memory!
    # Without this, the networkx graph grows indefinitely and slows down training!
    session.reset()
    
    if step % 10 == 0 or step == max_steps - 1:
        loss_val = float(np.asarray(loss.data))
        pbar.set_postfix({'Loss': f"{loss_val:.4f}"})

pbar.close()
print("✅ Extended Training Complete!")


# --- 4. Generation Evaluation with TQDM ---
def generate_visual(model, idx, max_new_tokens, block_size):
    print(f"\\n--- GENERATING {max_new_tokens} TOKENS ---\\n")
    gen_pbar = tqdm(range(max_new_tokens), desc="Generating Text")
    
    for _ in gen_pbar:
        idx_cond = idx[:, -block_size:] 
        logits = model(idx_cond)
        logits_last = np.asarray(logits.data)[:, -1, :] 
        
        v_max = np.max(logits_last, axis=-1, keepdims=True)
        exp_logits = np.exp(logits_last - v_max)
        probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        
        p = probs[0]
        next_token = np.random.choice(len(p), p=p)
        idx = np.append(idx, [[next_token]], axis=1)
        
        # ✅ Also reset session during generation!
        session.reset()
        
    gen_pbar.close()
    return idx

context = np.zeros((1, 1), dtype=np.int32)
generated_indices = generate_visual(big_model, context, max_new_tokens=300, block_size=block_size)

decode = lambda l: ''.join([chars[i] for i in l])
print("\\n--- GENERATED TEXT ---\\n")
print(decode(generated_indices[0]))


--- Initializing Larger Model ---
--- Starting Extended Training ---


Training:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 